# DD-PRiSM-plus — Build the L1000 drug signatures

Produces `drug_signatures_l1000.parquet`: one 978-landmark-gene transcriptomic
signature per drug, from **CMap / LINCS 2020** (the newer, larger release),
mapped to NSC by InChIKey. **CPU session, Internet On** — no GPU needed.

Attach two inputs via **Add Input**:

1. Your **01_setup_and_data** output — for `nsc_smiles.csv`.
2. A **Kaggle Dataset holding the big matrix** `level5_beta_trt_cp_n720216x12328.gctx`
   (for example the public *lincs-data* dataset). Attaching it as a Dataset keeps
   it in read-only `/kaggle/input`, off your working-disk quota.

The three small metadata tables are downloaded automatically from CLUE below —
you do **not** need to upload those.

In [ ]:
import os, glob, subprocess

REPO = '/kaggle/working/ddprism-plus'
OUT  = '/kaggle/working/l1000'

ok = subprocess.run(['curl', '-sI', '--max-time', '15', 'https://github.com'],
                    capture_output=True).returncode == 0
print('internet:', 'ON' if ok else 'OFF  <-- enable it, then rerun')

if os.path.exists(REPO):
    !cd {REPO} && git pull --quiet
else:
    !git clone --quiet https://github.com/SanaNiroomand/DD-PRiSM-plus.git {REPO}
os.chdir(REPO)
print('working in', os.getcwd())

## Install

`cmapPy` reads the `.gctx`; `rdkit` canonicalises SMILES to InChIKeys for the
NSC↔L1000 mapping.

In [ ]:
!pip install --quiet cmapPy rdkit pyarrow
print('installed')

## Gather the files

The three small CMap 2020 metadata tables are fetched from CLUE; the big
`.gctx` is found in whatever dataset you attached. Everything is linked into
one folder so the builder sees a single release together.

In [ ]:
import glob, os

LINCS = '/kaggle/working/lincs'
DATA  = '/kaggle/working/raw'
os.makedirs(LINCS, exist_ok=True)
os.makedirs(DATA, exist_ok=True)

CLUE = 'https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020'
for name in ('geneinfo_beta.txt', 'siginfo_beta.txt', 'compoundinfo_beta.txt'):
    target = os.path.join(LINCS, name)
    if not os.path.exists(target):
        !wget -q -O {target} {CLUE}/{name}
    print(name, os.path.getsize(target), 'bytes')

# the big matrix comes from the attached Kaggle dataset
gctx = glob.glob('/kaggle/input/**/level5_beta_trt_cp*.gctx', recursive=True)
if not gctx:
    raise SystemExit('level5_beta_trt_cp*.gctx not found -- attach a Kaggle '
                     'dataset that holds it via Add Input.')
link = os.path.join(LINCS, os.path.basename(gctx[0]))
if not os.path.exists(link):
    os.symlink(gctx[0], link)
print('gctx:', gctx[0])

# nsc_smiles.csv from the 01_setup_and_data output
smiles = glob.glob('/kaggle/input/**/nsc_smiles.csv', recursive=True)
if not smiles:
    raise SystemExit('nsc_smiles.csv not found -- attach the 01_setup_and_data '
                     'output via Add Input.')
dst = os.path.join(DATA, 'nsc_smiles.csv')
if not os.path.exists(dst):
    os.symlink(smiles[0], dst)
print('nsc_smiles:', smiles[0])

## Build the signatures

Reads only the 978 landmark rows and the signatures of drugs it can map to an
NSC, averages each drug's signatures into one consensus vector, and writes the
parquet. `--all-compounds` maps every NSC with a SMILES.

**Read the coverage line it prints** — it says how many of the drugs got an
L1000 signature. That is the honest scope of the whole L1000 experiment.

In [ ]:
!python scripts/fetch_l1000.py --lincs {LINCS} --data {DATA} --out {OUT} --all-compounds

## What came out

In [ ]:
import pandas as pd, os
frame = pd.read_parquet(os.path.join(OUT, 'drug_signatures_l1000.parquet'))
print('shape:', frame.shape)
print('drugs:', f'{len(frame):,}')
frame.iloc[:3, :6]

## Save it

**Save Version → Save & Run All (Commit).** The `08_experiment_l1000` and
`09_experiment_l1000_only` notebooks attach this output alongside the
`02_preprocess` output, via **Add Input → Your Work → Notebook Output**.